# 07 — Full Pipeline Verification

## Goal
Run the **complete detection pipeline** on every scenario and compare the output against known expected outcomes.  
This is our integration test — if all scenarios pass, the algorithm is correct.

---

## Detection Pipeline (Summary)

```
fixtures_enhanced.csv
       │
       ▼
 [1] is_base gate  →  body_ratio ≤ 0.50
       │
       ▼
 [2] find_base_clusters()  →  consecutive base candles → cluster list
       │
       ▼
 [3] evaluate_cluster()  →  width/ATR ≤ 2.5, compactness ≤ 0.80
       │
       ▼
 [4] detect_legs()  →  leg_net / ATR ≥ 1.5 for both legs
       │
       ▼
 [5] classify_formation()  →  RBR / DBD / DBR / RBD
       │
       ▼
 [6] evaluate_departure()  →  departure/ATR ≥ 1.5  AND  departure/zone_width ≥ 1.0
       │
       ▼
  Zone confirmed  (or rejected at any step above)
```

---

## Scenarios as Tests
Every scenario in `fixtures_labeled.csv` is a **hand-crafted test case**. The name encodes what it tests:

- `A_demand_RBR` → happy path, a textbook rally-base-rally
- `D_wide_base` → tests the width gate (should be rejected)
- `J_long_base` → tests the max-candle gate (should be rejected)

**Ground Truth** = what a human trader would mark on the chart.  
**Algorithm Result** = what the current pipeline actually outputs.  
A scenario passes when the two match.

---

## Expected Outcomes

| Scenario | What it tests | Ground Truth | Algorithm | Status |
|----------|---------------|--------------|-----------|--------|
| A_demand_RBR      | Clean RBR — happy path                       | Demand (RBR) | Demand (RBR) | ✓ pass |
| B_supply_DBD      | Clean DBD — happy path                       | Supply (DBD) | Supply (DBD) | ✓ pass |
| C_weak_dep        | Departure gate — move out is too small       | None         | None         | ✓ pass |
| D_wide_base       | Width gate — wide doji should be rejected    | None         | Demand       | ✗ false positive |
| E_doji_base       | Doji body counts as a base candle            | Demand (DBR) | Demand (DBR) | ✓ pass |
| F_fresh_vs_tested | Pullback after base — should still pass      | Demand (DBR) | None         | ✗ false negative |
| G_nested          | Nested base inside a larger one              | Demand (RBR) | None         | ✗ false negative |
| H_DBR             | Drop-base-rally formation                    | Demand (DBR) | Demand (DBR) | ✓ pass |
| I_RBD             | Rally-base-drop formation                    | Supply (RBD) | Supply (RBD) | ✓ pass |
| J_long_base       | Max-candle gate — base > 5 candles rejected  | None         | None         | ✓ pass |

**Score: 7 / 10.**  
- **D** — false positive: wide-doji base slips through because there's no per-candle range gate.
- **F** — false negative: an intermediate pullback shrinks the peak-excursion departure measurement.
- **G** — false negative: a nested inner base sits inside a larger one; outer legs don't qualify.

These three are the current pipeline's known limitations — later notebooks (freshness, nested zones, curve) address them.

---

## Visualization
The final chart renders all confirmed zones across the full dataset in a single view —  
green boxes = demand, red boxes = supply, each labelled with formation type and scenario name.


## 1. Constants — everything from NB04, NB05, NB06 in one place
This notebook is the **integration test**. It re-declares every threshold from the previous notebooks so it can run standalone, and so anyone reading it can see all the knobs side-by-side without flipping between files.


In [9]:
import pandas as pd
import numpy as np

# base-cluster
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80

# legs
LEG_CANDLES = 3
LEG_ATR_MIN = 1.5

# departure
DEPARTURE_CANDLES   = 3
DEPARTURE_ATR_MIN   = 1.5
DEPARTURE_RATIO_MIN = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)

o_arr   = df["open"].values
h_arr   = df["high"].values
l_arr   = df["low"].values
c_arr   = df["close"].values
atr_arr = df["atr"].values

print(f"{len(df)} candles loaded")


98 candles loaded


## 2. Base-cluster helpers
`find_base_clusters()` walks the dataset and groups consecutive `is_base` candles into `(start, end)` ranges, then `cluster_ok()` applies the width and compactness gates from NB04.


In [10]:
def find_base_clusters():
    clusters = []
    i = 0
    while i < len(df):
        if not df["is_base"].iloc[i]:
            i += 1
            continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]:
            j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES:
            clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs : be + 1].mean()
    width   = h_arr[bs : be + 1].max() - l_arr[bs : be + 1].min()
    compact = (c_arr[bs : be + 1].max() - c_arr[bs : be + 1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)


## 3. Leg classification and formation lookup
`classify_move(net, atr)` collapses a net price move into one of `up / down / flat` using `LEG_ATR_MIN`. `measure_legs(bs, be)` reads the 3 candles before the base and the 3 after, then returns `(leg_in_dir, leg_out_dir, avg_atr)`.

It returns `None` when there aren't enough candles on either side — the guard is **explicit** because Python's negative indexing would silently wrap around and look at the wrong end of the dataset.


In [11]:
def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    if net >=  t: return "up"
    if net <= -t: return "down"
    return "flat"

def measure_legs(bs, be):
    avg_atr = atr_arr[bs : be + 1].mean()

    if bs < LEG_CANDLES:
        return None
    lin_net = c_arr[bs - 1] - o_arr[bs - LEG_CANDLES]

    if be + LEG_CANDLES >= len(c_arr):
        return None
    lout_net = c_arr[be + LEG_CANDLES] - c_arr[be]

    return (classify_move(lin_net,  avg_atr),
            classify_move(lout_net, avg_atr),
            avg_atr)


## 4. Proximal / distal and the departure gate
Same definitions as NB06:

- **demand zone** → `proximal = base.high.max()`, `distal = base.low.min()`
- **supply zone** → `proximal = base.low.min()`,  `distal = base.high.max()`

Departure is measured against the **proximal** line — that is the edge price has to cross convincingly for the zone to be considered "live".


In [12]:
def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs : be + 1], l_arr[bs : be + 1]
    if zone_type == "demand":
        return bh.max(), bl.min()
    return bl.min(), bh.max()

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr) - 1, be + DEPARTURE_CANDLES)
    candles_h = h_arr[be + 1 : end + 1]
    candles_l = l_arr[be + 1 : end + 1]
    if len(candles_h) == 0:
        return 0, 0, 0, False
    dep       = (candles_h.max() - proximal) if zone_type == "demand" else (proximal - candles_l.min())
    dep_ratio = dep / zone_width if zone_width > 0 else 0
    dep_atr   = dep / avg_atr   if avg_atr   > 0 else 0
    passed    = (dep_ratio >= DEPARTURE_RATIO_MIN) and (dep_atr >= DEPARTURE_ATR_MIN)
    return round(dep, 3), round(dep_ratio, 2), round(dep_atr, 2), passed


## 5. `detect_zones()` — the full pipeline as a single function
Every gate from NB04 → NB06 strung together. A cluster has to survive **all** of them to make the final list:

1. `find_base_clusters()` — consecutive base candles
2. `cluster_ok()`         — width and compactness
3. `measure_legs()`       — enough candles around the base
4. `FORMATION_MAP`        — recognised in/out direction pair
5. `check_departure()`    — price actually left the zone


In [13]:
def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be):
            continue
        legs = measure_legs(bs, be)
        if legs is None:
            continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None:
            continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed:
            continue
        results.append(dict(
            bs=bs, be=be,
            formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected\n")
for z in zones:
    scen = labeled["scenario"].iloc[z["bs"]]
    print(f"  {z['zone_type']:6s} {z['formation']}  {scen:<20}  "
          f"prox={z['proximal']:.2f}  dist={z['distal']:.2f}  "
          f"dep_ratio={z['dep_ratio']}  dep_atr={z['dep_atr']}")


9 zones detected

  demand RBR  A_demand_RBR          prox=105.90  dist=104.90  dep_ratio=5.6  dep_atr=3.53
  supply RBD  A_demand_RBR          prox=109.00  dist=110.40  dep_ratio=2.14  dep_atr=1.72
  supply DBD  B_supply_DBD          prox=105.80  dist=106.70  dep_ratio=5.89  dep_atr=3.02
  demand RBR  D_wide_base           prox=115.80  dist=114.40  dep_ratio=3.64  dep_atr=2.28
  demand RBR  E_doji_base           prox=120.90  dist=120.00  dep_ratio=5.0  dep_atr=2.1
  demand RBR  E_doji_base           prox=125.40  dist=124.00  dep_ratio=3.86  dep_atr=2.49
  demand DBR  H_DBR                 prox=135.70  dist=134.70  dep_ratio=7.1  dep_atr=3.34
  demand RBR  H_DBR                 prox=140.10  dist=139.00  dep_ratio=5.0  dep_atr=2.51
  supply RBD  I_RBD                 prox=144.60  dist=145.60  dep_ratio=4.8  dep_atr=2.23


## 6. Expected outcomes per scenario
The fixtures CSV was hand-crafted with labelled scenarios A → J. For each scenario we know in advance whether the algorithm **should** find a zone, and which side (`demand` / `supply`) it should be. This is the ground truth the comparison table is checked against.


In [14]:
EXPECTED = {
    "A_demand_RBR":      ("demand", "should find"),
    "B_supply_DBD":      ("supply", "should find"),
    "C_weak_dep":        (None,     "should reject — departure too weak"),
    "D_wide_base":       (None,     "should reject — base too wide"),
    "E_doji_base":       ("demand", "should find — doji counts as base"),
    "F_fresh_vs_tested": ("demand", "should find"),
    "G_nested":          ("demand", "should find — inner base"),
    "H_DBR":             ("demand", "should find — DBR"),
    "I_RBD":             ("supply", "should find — RBD"),
    "J_long_base":       (None,     "should reject — too many candles"),
}


## 7. Comparison table — detected vs expected
We group the detected zones by the scenario label of their **base start** candle, then for each row in `EXPECTED` decide:

- expected `None` and nothing detected → **PASS** (correctly rejected)
- expected `None` but something detected → **FAIL** (false positive)
- expected a type and got a match → **PASS**
- expected a type and got nothing → **FAIL** (missed)


In [15]:
detected_by_scen = {}
for z in zones:
    scen = labeled["scenario"].iloc[z["bs"]]
    detected_by_scen.setdefault(scen, []).append(z)

print(f"{'Scenario':<22} {'Expected':^9} {'Detected':^9}  {'Status':<8}  Note")
print("-" * 80)

passed = failed = 0
for scen, (expected_type, note) in EXPECTED.items():
    found = detected_by_scen.get(scen, [])
    if expected_type is None:
        ok = (len(found) == 0)
        status = "PASS" if ok else "FAIL ← false positive"
    else:
        ok = (len(found) > 0)
        status = "PASS" if ok else "FAIL ← not detected"
    passed += int(ok)
    failed += int(not ok)
    det_str = found[0]["zone_type"] if found else "—"
    print(f"{scen:<22} {str(expected_type):^9} {det_str:^9}  {status:<8}  {note}")

print("-" * 80)
print(f"Result: {passed}/{passed + failed} scenarios pass")


Scenario               Expected  Detected   Status    Note
--------------------------------------------------------------------------------
A_demand_RBR            demand    demand    PASS      should find
B_supply_DBD            supply    supply    PASS      should find
C_weak_dep               None        —      PASS      should reject — departure too weak
D_wide_base              None     demand    FAIL ← false positive  should reject — base too wide
E_doji_base             demand    demand    PASS      should find — doji counts as base
F_fresh_vs_tested       demand       —      FAIL ← not detected  should find
G_nested                demand       —      FAIL ← not detected  should find — inner base
H_DBR                   demand    demand    PASS      should find — DBR
I_RBD                   supply    supply    PASS      should find — RBD
J_long_base              None        —      PASS      should reject — too many candles
--------------------------------------------------------

## 8. Visualization
Every detected zone drawn on the candlestick chart, colored by side (green = demand, red = supply). Proximal (dashed) and distal (dotted) lines extend into the departure window so the move out of the base is visible. Scenarios the algorithm was expected to find but missed are marked with a yellow arrow.


In [16]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"

FILL = {"demand": "rgba(38,166,154,0.15)", "supply": "rgba(239,83,80,0.15)"}
EDGE = {"demand": "#26a69a",               "supply": "#ef5350"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
))

for z in zones:
    x0  = df.index[z["bs"]]
    x1  = df.index[z["be"]]
    x1e = df.index[min(z["be"] + DEPARTURE_CANDLES + 1, len(df) - 1)]
    c_e = EDGE[z["zone_type"]]

    fig.add_vrect(x0=x0, x1=x1,
                  fillcolor=FILL[z["zone_type"]],
                  line=dict(color=c_e, width=1, dash="dot"))
    fig.add_shape(type="line", x0=x0, x1=x1e,
                  y0=z["proximal"], y1=z["proximal"],
                  line=dict(color=c_e, width=1, dash="dash"))
    fig.add_shape(type="line", x0=x0, x1=x1e,
                  y0=z["distal"], y1=z["distal"],
                  line=dict(color=c_e, width=1, dash="dot"))

    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']}<br>✓",
                       showarrow=False,
                       font=dict(size=8, color=c_e),
                       xanchor="left", yanchor="bottom")

# mark scenarios we expected but missed
for scen, (expected_type, _) in EXPECTED.items():
    if expected_type is None or scen in detected_by_scen:
        continue
    rows = labeled[labeled["scenario"] == scen]
    base_rows = rows[rows["note"].str.startswith("BASE")]
    if base_rows.empty:
        continue
    x_mid = base_rows.index[len(base_rows) // 2]
    y_mid = df.loc[base_rows.index, "high"].max() + 1
    fig.add_annotation(x=x_mid, y=y_mid,
                       text=f"missed<br>{scen.split('_',1)[-1]}",
                       showarrow=True, arrowhead=2,
                       font=dict(size=8, color="#ffeb3b"),
                       arrowcolor="#ffeb3b", ax=0, ay=-30)

fig.update_layout(
    title=f"Verify All Scenarios — {len(zones)} zones detected",
    height=600,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
